In [1]:
import os

In [2]:
%pwd

'd:\\Data Science\\project Series\\End_to_end_ReD_Wine_Quality_Mlops_Project\\research'

In [3]:
os.chdir('../')

In [44]:
from pathlib import Path
from dataclasses import dataclass

@dataclass(frozen=True)
class ModelTrainingConfig:
    root_dir: Path
    train_data_path: Path
    test_data_path: Path
    model_name: str
    n_estimators: int
    max_depth: int
    target_column: str



In [45]:
from WineQuality_Project.constants import *
from WineQuality_Project.utils.common import read_yaml,create_directories

In [46]:
from pathlib import Path

class ConfigManager:
    def __init__(
        self,
        config_filepath: Path = CONFIG_FILE_PATH,
        params_filepath: Path = PARAMS_FILE_PATH,
        schema_filepath: Path = SCHEMA_FILE_PATH
    ):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        self.schema = read_yaml(schema_filepath)

        create_directories([Path(self.config['artifact_root'])])

    def get_model_training_config(self) -> ModelTrainingConfig:
        config = self.config['model_training']
        params = self.params['model_trainer']

        create_directories([Path(config['root_dir'])])

        return ModelTrainingConfig(
            root_dir=Path(config['root_dir']),
            train_data_path=Path(config['train_data_path']),
            test_data_path=Path(config['test_data_path']),
            model_name=config['model_name'],
            n_estimators=params['n_estimators'],
            max_depth=params['max_depth'],
            target_column=self.schema['target_column']
        )


In [47]:
import os
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
import joblib


In [48]:

class ModelTraining:
    def __init__(self, config: ModelTrainingConfig):
        self.config = config

    def train(self):
        # Load data
        train_df = pd.read_csv(self.config.train_data_path)
        test_df = pd.read_csv(self.config.test_data_path)

        # Split features & target
        X_train = train_df.drop(self.config.target_column, axis=1)
        y_train = train_df[self.config.target_column]

        X_test = test_df.drop(self.config.target_column, axis=1)
        y_test = test_df[self.config.target_column]

        # Model
        model = RandomForestRegressor(
            n_estimators=self.config.n_estimators,
            max_depth=self.config.max_depth,
            random_state=42
        )

        # Train
        model.fit(X_train, y_train)

        # Save model
        model_path = os.path.join(
            self.config.root_dir,
            self.config.model_name
        )

        joblib.dump(model, model_path)

        return model


In [49]:
import logging
try:
    logging.info(">>> Model Training Stage Started <<<")

    config_manager = ConfigManager()
    model_training_config = config_manager.get_model_training_config()

    model_trainer = ModelTraining(config=model_training_config)
    model_trainer.train()

    logging.info(">>> Model Training Stage Completed Successfully <<<")

except Exception as e:
    logging.error("❌ Error occurred in Model Training Stage")
    logging.exception(e)
    raise e

[2025-12-12 20:40:04] [INFO] WineQualityLogger - YAML file: config\config.yml loaded successfully
INFO:WineQualityLogger:YAML file: config\config.yml loaded successfully
[2025-12-12 20:40:04] [INFO] WineQualityLogger - YAML file: params.yaml loaded successfully
INFO:WineQualityLogger:YAML file: params.yaml loaded successfully
[2025-12-12 20:40:04] [INFO] WineQualityLogger - YAML file: schema.yaml loaded successfully
INFO:WineQualityLogger:YAML file: schema.yaml loaded successfully
[2025-12-12 20:40:04] [INFO] WineQualityLogger - Directory created at: artifacts
INFO:WineQualityLogger:Directory created at: artifacts
[2025-12-12 20:40:04] [INFO] WineQualityLogger - Directory created at: artifacts\model_trainer
INFO:WineQualityLogger:Directory created at: artifacts\model_trainer
